In [1]:
import numpy as np

def price_binary_put(paths, strike, payout_amount):
    """
    Prices a Binary Put Option (All-or-Nothing).
    Pays 'payout_amount' if terminal price < strike, else 0.
    """
    # Terminal prices at Week 3
    terminal_prices = paths[-1, :]
    
    # Payoff is payout_amount where condition is met, 0 otherwise
    payoffs = np.where(terminal_prices < strike, payout_amount, 0.0)
    
    expected_price = np.mean(payoffs)
    std_error = np.std(payoffs) / np.sqrt(len(payoffs))
    return expected_price, std_error

def price_knockout_put(paths, strike, barrier):
    """
    Prices a Down-and-Out Put Option.
    Behaves like a regular put, but becomes worthless if the asset 
    EVER trades below the barrier during the simulation grid.
    """
    # Check the lowest price reached in each simulation path
    min_prices_reached = np.min(paths, axis=0)
    
    # Boolean array: True if path survived (never breached barrier), False if knocked out
    survived = min_prices_reached >= barrier
    
    # Calculate standard put payoffs at Week 3
    terminal_prices = paths[-1, :]
    standard_payoffs = np.maximum(strike - terminal_prices, 0)
    
    # Apply the knockout condition: 0 if breached, standard payoff if survived
    payoffs = np.where(survived, standard_payoffs, 0.0)
    
    expected_price = np.mean(payoffs)
    std_error = np.std(payoffs) / np.sqrt(len(payoffs))
    return expected_price, std_error

def price_chooser_option(paths, strike):
    """
    Prices a 3-Week Chooser Option.
    At Week 2, becomes a Call or Put based on which is In-The-Money.
    """
    days_per_week = 5
    steps_per_day = 4
    week_2_step = 2 * days_per_week * steps_per_day  # Step 40
    
    # Prices at decision time (Week 2) and expiry (Week 3)
    prices_w2 = paths[week_2_step, :]
    prices_w3 = paths[-1, :]
    
    # Decision Logic: Becomes a Call if S_w2 > Strike, otherwise a Put
    # (If exactly ATM at Week 2, both have 0 intrinsic value, so either choice is mathematically fine)
    chose_call = prices_w2 > strike
    
    # Calculate what the payoff WOULD be for both options at Week 3
    call_payoffs_w3 = np.maximum(prices_w3 - strike, 0)
    put_payoffs_w3 = np.maximum(strike - prices_w3, 0)
    
    # Assign the final payoff based on the choice made at Week 2
    payoffs = np.where(chose_call, call_payoffs_w3, put_payoffs_w3)
    
    expected_price = np.mean(payoffs)
    std_error = np.std(payoffs) / np.sqrt(len(payoffs))
    return expected_price, std_error

In [2]:
def price_vanilla_call(paths, strike, expiry_weeks):
    """
    Prices a European Call option.
    Payoff: max(S_T - K, 0) evaluated only at expiry.
    """
    days_per_week = 5
    steps_per_day = 4
    expiry_step = expiry_weeks * days_per_week * steps_per_day
    
    # Extract terminal prices at the exact expiry week
    terminal_prices = paths[expiry_step, :]
    
    # Calculate call payoffs
    payoffs = np.maximum(terminal_prices - strike, 0.0)
    
    expected_price = np.mean(payoffs)
    std_error = np.std(payoffs) / np.sqrt(len(payoffs))
    
    return expected_price, std_error

def price_vanilla_put(paths, strike, expiry_weeks):
    """
    Prices a European Put option.
    Payoff: max(K - S_T, 0) evaluated only at expiry.
    """
    days_per_week = 5
    steps_per_day = 4
    expiry_step = expiry_weeks * days_per_week * steps_per_day
    
    # Extract terminal prices at the exact expiry week
    terminal_prices = paths[expiry_step, :]
    
    # Calculate put payoffs
    payoffs = np.maximum(strike - terminal_prices, 0.0)
    
    expected_price = np.mean(payoffs)
    std_error = np.std(payoffs) / np.sqrt(len(payoffs))
    
    return expected_price, std_error

In [3]:
def generate_paths(initial_price, time_horizon_weeks, num_simulations=100000):
    """
    Generates Monte Carlo price paths for AETHER_CRYSTAL using Geometric Brownian Motion.
    
    Parameters:
    initial_price (float): The starting spot price (S0)
    time_horizon_weeks (int): The total weeks to simulate
    num_simulations (int): Number of paths to generate
    
    Returns:
    np.ndarray: Array of shape (total_steps + 1, num_simulations) containing the paths.
    """
    # Fixed challenge parameters
    sigma = 2.51           
    mu = 0.0               
    trading_days_yr = 252
    steps_per_day = 4
    days_per_week = 5
    
    # Calculate exact time steps
    dt = 1 / (trading_days_yr * steps_per_day)
    total_steps = time_horizon_weeks * days_per_week * steps_per_day
    
    # Generate random standard normal variables
    Z = np.random.standard_normal((total_steps, num_simulations))
    
    # Calculate the GBM drift and diffusion components
    drift = (mu - 0.5 * sigma**2) * dt
    diffusion = sigma * np.sqrt(dt) * Z
    
    # Compute step multipliers
    step_multipliers = np.exp(drift + diffusion)
    
    # Initialize the path array and apply cumulative product
    paths = np.zeros((total_steps + 1, num_simulations))
    paths[0] = initial_price
    paths[1:] = initial_price * np.cumprod(step_multipliers, axis=0)
    
    return paths

# Example usage:
paths = generate_paths(initial_price=50.0, time_horizon_weeks=3, num_simulations=100000)

In [4]:
# --- 1. Simulation Parameters ---
initial_price = 50.0
simulations = 1000000  
binary_payout = 10.0  # Set to your specified payout

print(f"Generating {simulations:,} Monte Carlo paths...")
paths = generate_paths(initial_price, time_horizon_weeks=3, num_simulations=simulations)
print("Paths generated. Pricing order book...\n")

# --- 2. Price Vanilla Calls ---
c50_2w, c50_2w_se = price_vanilla_call(paths, strike=50, expiry_weeks=2)
c50_3w, c50_3w_se = price_vanilla_call(paths, strike=50, expiry_weeks=3)
c60_3w, c60_3w_se = price_vanilla_call(paths, strike=60, expiry_weeks=3)

# --- 3. Price Vanilla Puts ---
p50_2w, p50_2w_se = price_vanilla_put(paths, strike=50, expiry_weeks=2)
p50_3w, p50_3w_se = price_vanilla_put(paths, strike=50, expiry_weeks=3)
p45_3w, p45_3w_se = price_vanilla_put(paths, strike=45, expiry_weeks=3)
p40_3w, p40_3w_se = price_vanilla_put(paths, strike=40, expiry_weeks=3)
p35_3w, p35_3w_se = price_vanilla_put(paths, strike=35, expiry_weeks=3)

# --- 4. Price Exotics ---
chooser, chooser_se = price_chooser_option(paths, strike=50)
binary, binary_se = price_binary_put(paths, strike=50, payout_amount=binary_payout)
knockout, knockout_se = price_knockout_put(paths, strike=45, barrier=35)

# --- 5. Output Final Order Book Valuations ---
print("="*45)
print("   AETHER_CRYSTAL THEORETICAL FAIR VALUES")
print("="*45)
print(f"Underlying (S0):   {initial_price}")
print(f"Volatility (σ):    251%\n")

print("--- VANILLA CALLS ---")
print(f"14-Day 50 Call:    {c50_2w:>6.2f} ± {c50_2w_se:.2f}")
print(f"21-Day 50 Call:    {c50_3w:>6.2f} ± {c50_3w_se:.2f}")
print(f"21-Day 60 Call:    {c60_3w:>6.2f} ± {c60_3w_se:.2f}\n")

print("--- VANILLA PUTS ---")
print(f"14-Day 50 Put:     {p50_2w:>6.2f} ± {p50_2w_se:.2f}")
print(f"21-Day 50 Put:     {p50_3w:>6.2f} ± {p50_3w_se:.2f}")
print(f"21-Day 45 Put:     {p45_3w:>6.2f} ± {p45_3w_se:.2f}")
print(f"21-Day 40 Put:     {p40_3w:>6.2f} ± {p40_3w_se:.2f}")
print(f"21-Day 35 Put:     {p35_3w:>6.2f} ± {p35_3w_se:.2f}\n")

print("--- EXOTIC OPTIONS ---")
print(f"Chooser (K=50):                 {chooser:>6.2f} ± {chooser_se:.2f}")
print(f"Binary Put (K=50, Payout={int(binary_payout)}):  {binary:>6.2f} ± {binary_se:.2f}")
print(f"Knockout Put (K=45, Barrier=35): {knockout:>6.2f} ± {knockout_se:.2f}")
print("="*45)

Generating 1,000,000 Monte Carlo paths...
Paths generated. Pricing order book...

   AETHER_CRYSTAL THEORETICAL FAIR VALUES
Underlying (S0):   50.0
Volatility (σ):    251%

--- VANILLA CALLS ---
14-Day 50 Call:      9.87 ± 0.02
21-Day 50 Call:     12.01 ± 0.03
21-Day 60 Call:      8.78 ± 0.02

--- VANILLA PUTS ---
14-Day 50 Put:       9.87 ± 0.01
21-Day 50 Put:      12.03 ± 0.01
21-Day 45 Put:       9.10 ± 0.01
21-Day 40 Put:       6.51 ± 0.01
21-Day 35 Put:       4.34 ± 0.01

--- EXOTIC OPTIONS ---
Chooser (K=50):                  21.89 ± 0.02
Binary Put (K=50, Payout=10):    6.20 ± 0.00
Knockout Put (K=45, Barrier=35):   0.21 ± 0.00
